# 🏗️ Notebook Guide — Protein Structural Biology & ML

## Learning Objectives
- [ ] Parse PDB files manually (fixed-width format, Atom dataclass)
- [ ] Compute RMSD between two protein structures
- [ ] Implement Kabsch algorithm (SVD-based optimal superposition)
- [ ] Build contact maps from pairwise Cα distances
- [ ] Compute phi/psi dihedral angles and classify secondary structure
- [ ] Build per-residue feature vectors for ML
- [ ] Understand TM-score and its advantages over RMSD

## Why Structural Biology for HackerRank?
Structural bioinformatics roles at pharma/biotech companies (Schrödinger, D.E. Shaw, Relay Therapeutics, etc.) test:
1. PDB parsing and manipulation
2. Structure comparison metrics (RMSD, TM-score, GDT-TS)
3. Feature engineering from 3D coordinates
4. Graph-based representations of protein structure

---

## 🤖 Claude Integration — Copy These Prompts

**For PDB Parsing:**
```
Explain PDB file format for bioinformatics.
What are ATOM records? What information is in each column (fixed-width format)?
Columns: atom_serial (7-11), atom_name (13-16), residue_name (18-20),
         chain_id (22), residue_seq (23-26), x (31-38), y (39-46), z (47-54)
How do I extract only Cα atoms? (atom_name == 'CA')
What's the difference between ATOM and HETATM records?
Show me a minimal PDB parser using Python dataclasses.
```

**For RMSD:**
```
Explain RMSD (Root Mean Square Deviation) for protein structure comparison.
Formula: RMSD = sqrt(1/N * Σ ||ri - ri'||²)
Why is RMSD sensitive to outliers?
What's a "good" RMSD for protein structure comparison?
How do I align structures before computing RMSD?
Is RMSD symmetric? (yes: RMSD(A,B) = RMSD(B,A))
```

**For the Kabsch Algorithm:**
```
Explain the Kabsch algorithm for optimal superposition of two point sets.
Walk me through the steps:
1. Translate both structures to their centroids
2. Compute the covariance matrix H = P.T @ Q
3. SVD: H = U @ Σ @ V.T
4. Check determinant (handle reflection)
5. Rotation matrix R = V @ U.T
Why is SVD the key step? What does it minimize?
Show me a numpy implementation.
```

**For Contact Maps:**
```
Explain protein contact maps.
What is a contact? (two residues whose Cα atoms are within 8Å)
What does a contact map look like as a matrix?
What structural features appear as patterns in the contact map?
(helix = near-diagonal contacts, beta sheet = off-diagonal parallel/antiparallel lines)
How do contact maps relate to AlphaFold2's distogram prediction?
```

**For Dihedral Angles (phi/psi):**
```
Explain phi and psi backbone dihedral angles in proteins.
What 4 atoms define phi? (C_{i-1}, N_i, Cα_i, C_i)
What 4 atoms define psi? (N_i, Cα_i, C_i, N_{i+1})
How do I compute the dihedral angle from 4 3D coordinates using cross products?
What are the phi/psi ranges for alpha helix? beta sheet? (Ramachandran plot regions)
```

**For TM-score:**
```
Explain TM-score and why it's better than RMSD for protein structure comparison.
Key properties:
- TM-score is length-normalized (0 to 1)
- TM-score > 0.5 suggests same fold
- RMSD is not normalized — a large protein with small RMSD can still be very different
Show me the formula including d0 (length-dependent normalization factor).
What's GDT-TS and how does it compare to TM-score?
```

---

## Structure Analysis Pipeline

```
PDB file
    │ parse_pdb()
    ▼
List[Atom(x, y, z, residue, chain, ...)]
    │
    ├──→ extract Cα atoms → [np.array(x,y,z)] × N
    │         │
    │         ├──→ pairwise distances → Contact Map (N×N matrix)
    │         ├──→ RMSD vs reference
    │         └──→ Kabsch superposition → optimal rotation matrix R
    │
    └──→ backbone N, Cα, C atoms → dihedral angles (phi, psi)
              │
              └──→ secondary structure classification
```

## Key Thresholds to Remember

| Metric | Threshold | Meaning |
|--------|-----------|---------|
| Contact distance | 8 Å | Cα-Cα contact |
| RMSD (good model) | < 2 Å | Very good structural similarity |
| TM-score (same fold) | > 0.5 | Likely same fold |
| Secondary structure | α-helix: φ≈-60°, ψ≈-40° | Ramachandran plot |
| Secondary structure | β-sheet: φ≈-120°, ψ≈+120° | Ramachandran plot |

---

## Interview Tip Bank
> **RMSD vs TM-score in interviews**: "RMSD is sensitive to a few large deviations and grows with protein length. TM-score normalizes by length and is less sensitive to outliers — better for comparing structures of different sizes."

> **Kabsch vs quaternion**: Both solve optimal superposition. Kabsch uses SVD (numpy-friendly). Quaternion methods are used in some MD software. Know Kabsch for interviews.

> **Contact map eigenvalues**: The eigenspectrum of a contact map encodes structural information. The first eigenvector distinguishes residues in helix vs loop vs sheet regions.

> **PDB format pitfall**: Columns are FIXED WIDTH, not space-separated. Use `line[30:38]`, not `line.split()` for coordinates.

---

## Self-Check
1. What columns in a PDB ATOM record contain x, y, z coordinates?
2. Write the RMSD formula. What does it measure geometrically?
3. In Kabsch algorithm, why do we check `det(V @ U.T)` before computing R?
4. A contact map has a long off-diagonal stripe parallel to the main diagonal. What structure does this suggest?
5. What are the phi/psi angles for a residue in an alpha helix?


## TL;DR — Plain English

**What this notebook does:** Teaches you to work directly with 3D protein structure files (PDB format) and measure how similar or different two structures are.

**After this notebook you can:**
- Parse a PDB file and extract atom coordinates using Biopython
- Calculate RMSD (root mean square deviation) between two protein structures
- Apply the Kabsch algorithm to optimally superimpose two structures before comparing
- Compute TM-score (topology match score) and interpret Ramachandran plots

**Why this matters:**
- HackerRank: Structure comparison questions appear in advanced bioinformatics assessments; RMSD is one of the most common interview calculation questions
- computational biology ML teams: AlphaFold3 is evaluated using RMSD, TM-score, and FAPE — you cannot discuss the model's performance without understanding these metrics

**Time:** ~3 hours | **Difficulty:** ⭐⭐⭐ | **Prerequisites:** 00/01 (Python & NumPy), basic chemistry helpful

## Protein Structure — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **PDB (Protein Data Bank)** | World database of experimentally determined 3D protein structures — free at rcsb.org |
| **PDB ID** | 4-character code for a structure (e.g. 1TIM) — like an ISBN for a protein structure |
| **atom coordinates** | X, Y, Z positions of each atom in Angstroms (1A = 0.1 nanometre) |
| **residue** | One amino acid in the protein chain — the repeating unit of the polymer |
| **chain** | One continuous polypeptide strand in a structure; multi-chain structures have A, B, C... |
| **Calpha (C-alpha)** | The central carbon of each amino acid backbone — simplest single-point residue representation |
| **RMSD** | Root Mean Square Deviation — average distance between equivalent atoms in two structures; 0 = identical |
| **B-factor** | Temperature factor — measures how much an atom moves/vibrates; high B = flexible/uncertain |
| **secondary structure** | Local patterns: alpha-helix (coiled), beta-sheet (flat), or loop/coil (irregular) |
| **TM-score** | Template Modelling score (0-1): >0.5 = same fold, >0.7 = very similar, 1.0 = identical |
| **Kabsch algorithm** | Optimal rotation that minimises RMSD between two sets of coordinates |
| **cryo-EM** | Electron microscopy method for solving structures of large complexes without crystals |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students who are new to structural biology but comfortable with arrays and geometry.

**How to study it on a first pass:**
- draw the geometric picture first
- then inspect coordinates, matrices, and rotations
- always connect each metric to the structural question it answers

**Common traps:** using RMSD without understanding alignment, confusing coordinate representation with biology, and skipping visual intuition.


## Canonical Resource Upgrade

Use these to build intuition:
- [RCSB PDB-101](https://pdb101.rcsb.org/learn/guide-to-understanding-pdb-data)
- [wwPDB format documentation](https://www.wwpdb.org/documentation/file-format)
- [PyMOLWiki](https://pymolwiki.org/)


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **Sequence alignment** — RMSD compares 3D structures like alignment compares 1D sequences. *Review: `01_sequence_analysis/01_alignment_algorithms.ipynb`*
- **Linear algebra** — rotation matrices, eigenvalues, SVD (used in Kabsch superposition). *Review: `00_python_ml_basics/08_mathematical_foundations.ipynb`*
- **NumPy array operations** — broadcasting, matrix multiplication, indexing. *Review: `00_python_ml_basics/01_python_core_for_bioinformatics.ipynb`*

**Quick recap of terms used in this notebook:**
- **PDB file** — Protein Data Bank format storing 3D atomic coordinates as fixed-width text (ATOM records at columns 31-54 for x,y,z)
- **RMSD** — Root Mean Square Deviation: average distance between paired atoms after optimal superposition; lower = more similar
- **Kabsch algorithm** — finds the optimal rotation matrix to minimise RMSD between two sets of 3D points using SVD
- **TM-score** — template modelling score (0-1) that is length-normalised, unlike RMSD; TM > 0.5 generally means same fold
- **Ramachandran plot** — scatter plot of backbone phi/psi dihedral angles; valid proteins cluster in specific regions


## What This Notebook Teaches (Plain English)

**Proteins are 3D machines** made of amino acid chains. Their shape determines their function — a misfolded protein can cause disease (Alzheimer's, Parkinson's). This notebook teaches you to work with protein 3D structure data computationally.

### Key Concepts Before You Start

| Concept | Plain English |
|---------|--------------|
| **Protein** | A chain of amino acids (20 types) that folds into a 3D shape |
| **Residue** | One amino acid unit in the chain |
| **Atom** | A single atom (C, N, O, S, H) within a residue |
| **Cα (C-alpha)** | The central backbone carbon of each amino acid — the simplest representation of protein shape |
| **PDB file** | Text file storing 3D coordinates of every atom, downloaded from rcsb.org |
| **RMSD** | How far apart two structures are on average (in Ångströms, 1Å = 0.1nm). Lower = more similar |
| **TM-score** | 0-to-1 similarity score. > 0.5 means same protein fold family |
| **Contact map** | Matrix showing which residues are close in 3D space (< 8Å apart) |
| **Dihedral angles** | Two backbone angles (φ phi, ψ psi) that define how the chain bends at each residue |

### Why This Matters
Structure determines function. If you can compare structures (RMSD, TM-score), you can:
- Find evolutionarily related proteins
- Evaluate AlphaFold predictions vs experimental structures
- Design new drugs that fit a protein binding site

### What You Need Before Starting
- Python + NumPy (Notebooks 00/01 and 00/02)
- No structural biology background required — concepts explained inline

# Protein Structure & Structural Biology
**HackerRank Prep — Theme 3**

Covers: PDB parsing, RMSD/TM-score, secondary structure, contact maps, structural features for ML.

---

## 1. PDB File Parsing (No External Libraries)

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
import warnings
warnings.filterwarnings('ignore')

# Generic structural biology analysis
print("Structural Biology Analysis")
print("Key tools: Biopython PDBParser, RMSD, Kabsch algorithm")
print("Key metrics: TM-score, GDT_TS, lDDT, contact map F1")
print()
print("Standard distance thresholds:")
print("  CA-CA bond: ~3.8 Å")
print("  Contact map: 8.0 Å")
print("  H-bond: 3.5 Å")
print()
np.random.seed(42)
coords = np.random.randn(50, 3) * 10
print(f"Example structure: {len(coords)} CA atoms")
print(f"Center of mass: {coords.mean(axis=0).round(2)}")
print(f"Radius of gyration: {np.sqrt(((coords - coords.mean(axis=0))**2).sum(axis=1).mean()):.2f} Å")

> **Expected output:** Header text: `Structural Biology Analysis` with key tools and metrics listed  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

## 2. RMSD Calculation

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
import warnings
warnings.filterwarnings('ignore')

# Generic structural biology analysis
print("Structural Biology Analysis")
print("Key tools: Biopython PDBParser, RMSD, Kabsch algorithm")
print("Key metrics: TM-score, GDT_TS, lDDT, contact map F1")
print()
print("Standard distance thresholds:")
print("  CA-CA bond: ~3.8 Å")
print("  Contact map: 8.0 Å")
print("  H-bond: 3.5 Å")
print()
np.random.seed(42)
coords = np.random.randn(50, 3) * 10
print(f"Example structure: {len(coords)} CA atoms")
print(f"Center of mass: {coords.mean(axis=0).round(2)}")
print(f"Radius of gyration: {np.sqrt(((coords - coords.mean(axis=0))**2).sum(axis=1).mean()):.2f} Å")

## 3. Contact Map

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
import warnings
warnings.filterwarnings('ignore')

# Generic structural biology analysis
print("Structural Biology Analysis")
print("Key tools: Biopython PDBParser, RMSD, Kabsch algorithm")
print("Key metrics: TM-score, GDT_TS, lDDT, contact map F1")
print()
print("Standard distance thresholds:")
print("  CA-CA bond: ~3.8 Å")
print("  Contact map: 8.0 Å")
print("  H-bond: 3.5 Å")
print()
np.random.seed(42)
coords = np.random.randn(50, 3) * 10
print(f"Example structure: {len(coords)} CA atoms")
print(f"Center of mass: {coords.mean(axis=0).round(2)}")
print(f"Radius of gyration: {np.sqrt(((coords - coords.mean(axis=0))**2).sum(axis=1).mean()):.2f} Å")

## 4. Secondary Structure Features

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
import warnings
warnings.filterwarnings('ignore')

# Generic structural biology analysis
print("Structural Biology Analysis")
print("Key tools: Biopython PDBParser, RMSD, Kabsch algorithm")
print("Key metrics: TM-score, GDT_TS, lDDT, contact map F1")
print()
print("Standard distance thresholds:")
print("  CA-CA bond: ~3.8 Å")
print("  Contact map: 8.0 Å")
print("  H-bond: 3.5 Å")
print()
np.random.seed(42)
coords = np.random.randn(50, 3) * 10
print(f"Example structure: {len(coords)} CA atoms")
print(f"Center of mass: {coords.mean(axis=0).round(2)}")
print(f"Radius of gyration: {np.sqrt(((coords - coords.mean(axis=0))**2).sum(axis=1).mean()):.2f} Å")

## 5. Structural Features for ML

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
import warnings
warnings.filterwarnings('ignore')

# Generic structural biology analysis
print("Structural Biology Analysis")
print("Key tools: Biopython PDBParser, RMSD, Kabsch algorithm")
print("Key metrics: TM-score, GDT_TS, lDDT, contact map F1")
print()
print("Standard distance thresholds:")
print("  CA-CA bond: ~3.8 Å")
print("  Contact map: 8.0 Å")
print("  H-bond: 3.5 Å")
print()
np.random.seed(42)
coords = np.random.randn(50, 3) * 10
print(f"Example structure: {len(coords)} CA atoms")
print(f"Center of mass: {coords.mean(axis=0).round(2)}")
print(f"Radius of gyration: {np.sqrt(((coords - coords.mean(axis=0))**2).sum(axis=1).mean()):.2f} Å")

## 6. TM-Score (Template Modeling Score)

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
import warnings
warnings.filterwarnings('ignore')

# Generic structural biology analysis
print("Structural Biology Analysis")
print("Key tools: Biopython PDBParser, RMSD, Kabsch algorithm")
print("Key metrics: TM-score, GDT_TS, lDDT, contact map F1")
print()
print("Standard distance thresholds:")
print("  CA-CA bond: ~3.8 Å")
print("  Contact map: 8.0 Å")
print("  H-bond: 3.5 Å")
print()
np.random.seed(42)
coords = np.random.randn(50, 3) * 10
print(f"Example structure: {len(coords)} CA atoms")
print(f"Center of mass: {coords.mean(axis=0).round(2)}")
print(f"Radius of gyration: {np.sqrt(((coords - coords.mean(axis=0))**2).sum(axis=1).mean()):.2f} Å")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- Protein structure levels (primary → quaternary), coordinate geometry in 3D space
- Rotation matrices, quaternions, and the Kabsch algorithm derivation for optimal superposition
- Ramachandran map theory — allowed φ/ψ dihedral angles, steric clashes, secondary structure regions
- TM-score vs RMSD — length-independent alignment scoring, GDT_TS, lDDT metrics
- [Introduction to Protein Structure — Branden & Tooze](https://www.routledge.com/Introduction-to-Protein-Structure/Branden-Tooze/p/book/9780815323051) (classic textbook)
- [PDB format specification](https://www.wwpdb.org/documentation/file-format) (ATOM/HETATM records)

### 2️⃣ Must-Have Popular Resources
- 📘 **Introduction to Protein Structure** (Branden & Tooze) — the canonical structural biology text
- 🎓 **PyMOL tutorials** (Schrödinger) — [pymolwiki.org](https://www.pymolwiki.org/index.php/Main_Page) (free, visual structure exploration)
- ⭐ **GitHub** [MDAnalysis/MDAnalysis](https://github.com/MDAnalysis/MDAnalysis) 1.1k★ — MD trajectory analysis, RMSD, RMSF, contacts
- ⭐ **GitHub** [biotite-dev/biotite](https://github.com/biotite-dev/biotite) 500★ — pure-Python structural biology (no PyMOL dependency)
- ⭐ **GitHub** [rdkit/rdkit](https://github.com/rdkit/rdkit) 2.5k★ — cheminformatics + small molecule 3D analysis
- 🤗 **HuggingFace** [facebook/esmfold_v1](https://huggingface.co/facebook/esmfold_v1) — single-sequence structure prediction in one forward pass
- 📊 **Kaggle** [PDB Secondary Structure](https://www.kaggle.com/datasets/alfrandom/protein-secondary-structure) — 200k+ chains with DSSP labels

### 3️⃣ Practicals — Hands-On
- 💻 **Exercise**: Parse 10 PDB structures manually — extract ATOM records, build Cα coordinate arrays
- 💻 **Exercise**: Compute all-vs-all pairwise RMSD matrix for a set of related structures (e.g., lysozyme variants)
- 💻 **Exercise**: Plot Ramachandran diagram for each structure — color by secondary structure element
- 💻 **Exercise**: Implement Kabsch superposition from scratch (SVD-based rotation matrix)
- 🔗 **GitHub** [foldseek](https://github.com/steineggerlab/foldseek) — structural alignment at 100x Dali speed (study the approach)
- 📊 **Kaggle** [Protein Structure Prediction](https://www.kaggle.com/competitions/stanford-ribonanza-rna-folding) — RNA structure challenge (similar geometry concepts)
- 🤗 **HuggingFace Space** [ESMFold demo](https://huggingface.co/spaces/facebook/ESMFold) — predict structures in browser

### 4️⃣ Real-World Problems
- 🏭 **Industry**: Drug discovery structure-based design — Schrödinger, OpenEye, RxBio pipeline
- 🏭 **Industry**: Cryo-EM analysis at structural genomics consortia — EMDB deposition, model building
- 🏭 **Industry**: Antibody humanization — CDR loop RMSD, sequence-structure co-optimization
- 📊 **Dataset**: [RCSB PDB](https://www.rcsb.org/) — 210k+ experimental structures with resolution, R-factors
- 📊 **Dataset**: [CATH/SCOP](https://www.cathdb.info/) — domain classification, hierarchical fold families
- 🔬 **Application**: Protein–protein docking — shape complementarity, electrostatics, scoring functions

### 5️⃣ Interview Tips
- ❓ **Must know**: RMSD vs TM-score — why RMSD is sensitive to outliers and alignment length, when TM-score is preferred
- ❓ **Must know**: Kabsch vs quaternion superposition — both find optimal rotation, quaternions avoid gimbal lock
- ❓ **Must know**: What pLDDT 70 means in AlphaFold2 — "confident" region (>70), meaning of very low (<50) regions (disordered)
- ⚠️ **Common mistake**: Comparing RMSD without first superposing structures — always align before computing distances
- ⚠️ **Common mistake**: Including HETATM records (ligands/water) in Cα RMSD calculation — filter ATOM + CA only
- 💡 **Pro tip**: For contact maps, threshold at 8Å Cβ–Cβ distance (not Cα) for better sidechain sensitivity
- 💡 **Pro tip**: Ramachandran outliers (>2% of residues) are a structure quality red flag — mention this in interviews

### 6️⃣ Hot Industry Topics
- 🔥 **Trending** [foldseek](https://github.com/steineggerlab/foldseek) 2k★ — structural search 100x faster than Dali using 3Di alphabet
- 🔥 **Trending** [PeSTo](https://github.com/LBM-EPFL/PeSTo) — protein–ligand interface prediction, geometric deep learning
- 🔥 **Trending** [chai-lab/chai-1](https://github.com/chaidiscovery/chai-1) — open-source AlphaFold3 alternative (Chai-1), handles proteins+ligands+nucleic acids
- 🔥 **Trending** AlphaFold3 + MD hybrid workflows — predict structure → run MD for dynamics → validate with experimental HDX-MS
- 🚀 **Build this**: RMSD-based conformational clustering — load 50 NMR models from one PDB entry, pairwise RMSD matrix, Ward linkage clustering, visualize dendrogram
- 🚀 **Build this**: Ramachandran quality checker — parse PDB, compute φ/ψ for all residues, flag outliers, report per-chain statistics

### Time Complexity — Structural Biology Operations
| Operation | Time | Space | Notes |
|-----------|------|-------|-------|
| PDB parsing (n atoms) | O(n) | O(n) | Biopython BioPDB |
| RMSD (Kabsch, n atoms) | O(n) after SVD | O(n) | SVD is O(n³) but n=few hundred |
| TM-score | O(n²) | O(n) | pairwise distances |
| Ramachandran angles | O(n) | O(n) | one pass |
| Contact map | O(n²) | O(n²) | all pairwise Cα distances |
| Solvent accessibility (SASA) | O(n² × probes) | O(n) | Lee-Richards algorithm |
| Secondary structure assignment | O(n) | O(n) | DSSP |
| Hydrogen bond detection | O(n²) | O(bonds) | pairwise donor-acceptor |

In [ ]:
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import PPBuilder
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Simulate PDB parsing (minimal atom data)
pdb_str = """ATOM      1  N   ALA A   1       1.000   2.000   3.000  1.00 10.00           N
ATOM      2  CA  ALA A   1       2.000   3.000   4.000  1.00 10.00           C
ATOM      3  C   ALA A   1       3.000   4.000   5.000  1.00 10.00           C
ATOM      4  O   ALA A   1       4.000   5.000   6.000  1.00 10.00           O
ATOM      5  N   GLY A   2       5.000   6.000   7.000  1.00 10.00           N
ATOM      6  CA  GLY A   2       6.000   7.000   8.000  1.00 10.00           C
END
"""

import io
parser = PDBParser(QUIET=True)
structure = parser.get_structure('test', io.StringIO(pdb_str))

for model in structure:
    for chain in model:
        print(f"Chain {chain.id}:")
        for residue in chain:
            atoms = list(residue.get_atoms())
            print(f"  Residue {residue.resname} {residue.id[1]}: {len(atoms)} atoms")
            for atom in atoms[:2]:
                print(f"    {atom.name}: {atom.get_vector()}")

# 🌍 Real World Problems — Protein Structural Biology

---

## Problem 1 — Compare AlphaFold2 Prediction vs Experimental PDB Structure
**Dataset**: PDB `1TUP` (TP53 DBD, experimental) vs AlphaFold DB prediction
**Source**: [AlphaFold DB](https://alphafold.ebi.ac.uk/entry/P04637) | [PDB 1TUP](https://www.rcsb.org/structure/1TUP)
**Skills**: PDB parsing, RMSD, Kabsch superposition

```python
import urllib.request

def fetch_pdb(pdb_id: str) -> str:
    """Download PDB file from RCSB."""
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    with urllib.request.urlopen(url) as r:
        return r.read().decode()

# Download TP53 DNA-binding domain (chain A only, residues 94-292)
# pdb_text = fetch_pdb('1TUP')
# atoms = parse_pdb(pdb_text)
# ca_atoms = [a for a in atoms if a.name == 'CA' and a.chain == 'A']

# Download AlphaFold prediction for TP53 (P04637)
# af_url = "https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v4.pdb"

# YOUR TASK:
# 1. Parse both PDB files using parse_pdb() from this notebook
# 2. Extract matching Cα atoms (same residue numbers)
# 3. Compute RMSD before alignment
# 4. Apply Kabsch algorithm to get optimal superposition
# 5. Compute RMSD after alignment
# 6. Which regions have the highest per-residue deviation? (likely loops)

# Mock example to run locally:
import numpy as np
np.random.seed(42)
n_residues = 100
experimental = np.random.randn(n_residues, 3) * 3
predicted = experimental + np.random.randn(n_residues, 3) * 0.5  # good prediction

def rmsd(P, Q):
    return np.sqrt(np.mean(np.sum((P - Q)**2, axis=1)))

print(f"RMSD before alignment: {rmsd(experimental, predicted):.3f} Å")
# Apply kabsch_align() from this notebook, then compute again
```

**Real impact**: This is exactly what the AlphaFold paper (Jumper et al., Nature 2021) benchmarked — comparing predicted structures to experimental ones. Industry standard at pharma companies.

---

## Problem 2 — Drug Binding Site Detection via Contact Map Analysis
**Dataset**: PDB structures of kinase-inhibitor complexes (e.g., `3PP0` = EGFR + erlotinib)
**GitHub**: [github.com/deepmind/alphafold](https://github.com/deepmind/alphafold)
**Skills**: Contact maps, distance thresholding, feature vectors

```python
# Identify which residues are in the binding pocket of a drug
# Binding pocket = residues within 5Å of any ligand atom

def find_binding_pocket(protein_atoms: list, ligand_atoms: list,
                         threshold: float = 5.0) -> list[int]:
    """
    Find protein residue numbers within threshold Angstroms of any ligand atom.
    protein_atoms: list of Atom objects (from parse_pdb)
    ligand_atoms:  list of Atom objects (HETATM records for the drug)
    Returns: sorted list of residue numbers forming the binding pocket
    """
    import numpy as np
    pocket_residues = set()
    for pa in protein_atoms:
        if pa.name != 'CA': continue  # only Cα for speed
        for la in ligand_atoms:
            dist = np.linalg.norm(pa.coord - la.coord)
            if dist < threshold:
                pocket_residues.add(pa.res_seq)
                break
    return sorted(pocket_residues)

# YOUR TASK:
# 1. Parse PDB 3PP0 (EGFR + erlotinib)
# 2. Separate ATOM records (protein) from HETATM records (ligand)
# 3. Find binding pocket residues (within 5Å of erlotinib)
# 4. Cross-check with published binding site (EGFR ATP binding pocket: T790, L858...)
# 5. Build a contact map of ONLY the binding pocket residues

print("Binding pocket analysis requires downloading PDB 3PP0")
print("Try: fetch_pdb('3PP0') then parse ATOM vs HETATM lines")
```

---

## Problem 3 — Protein Stability Prediction from Sequence + Structure Features
**Dataset**: [Kaggle: Novozymes Enzyme Stability Prediction](https://www.kaggle.com/competitions/novozymes-enzyme-stability-prediction)
**Hugging Face**: [facebook/esm2_t6_8M_UR50D](https://huggingface.co/facebook/esm2_t6_8M_UR50D)
**Skills**: PDB featurization, RMSD, contact maps, ML

```python
# This is the Kaggle Novozymes competition (2022, prize: $50K)
# Task: predict thermostability (melting temperature Tm) of enzyme variants
# Input: protein sequence + PDB structure
# Output: Tm value (regression)

# Feature engineering approach (no deep learning needed to get top 30%):
def extract_structural_features(atoms: list) -> dict:
    """
    Extract ML features from a parsed PDB file.
    These features are predictive of protein stability.
    """
    import numpy as np
    ca_coords = np.array([a.coord for a in atoms if a.name == 'CA'])
    n = len(ca_coords)
    
    # Contact density (more contacts = more stable)
    dists = np.sqrt(((ca_coords[:, None] - ca_coords[None, :])**2).sum(axis=2))
    contact_matrix = (dists < 8.0) & (dists > 0)
    contact_density = contact_matrix.sum(axis=1).mean()
    
    # Radius of gyration (compactness)
    centroid = ca_coords.mean(axis=0)
    rg = np.sqrt(((ca_coords - centroid)**2).sum(axis=1).mean())
    
    return {
        'n_residues': n,
        'contact_density': contact_density,
        'radius_of_gyration': rg,
    }

# Then combine with sequence features (AA composition, PSSM, etc.)
# and train a regression model (GBM, Ridge) to predict Tm
print("Kaggle competition: kaggle.com/competitions/novozymes-enzyme-stability-prediction")
print("Hugging Face ESM2: huggingface.co/facebook/esm2_t6_8M_UR50D")
```

---

## Resources
| Resource | URL | Purpose |
|----------|-----|---------|
| RCSB PDB | rcsb.org | Download experimental structures |
| AlphaFold DB | alphafold.ebi.ac.uk | Download AF2 predicted structures |
| Kaggle: Enzyme Stability | kaggle.com/competitions/novozymes-enzyme-stability-prediction | Real competition |
| GitHub: AlphaFold | github.com/deepmind/alphafold | Source code |
| HF: ESM2 models | huggingface.co/facebook/esm2_t6_8M_UR50D | Protein embeddings |
| PDBe-KB | pdbe-kb.org | Functional annotations |

## Mastery Check

Before moving on, make sure you can:
1. explain what RMSD and TM-score measure
2. say why Kabsch alignment is needed before comparing structures
3. read a simple PDB-derived coordinate example
4. connect a structural metric to a biological interpretation


---
## 🔬 Real Structure Analysis — Actual PDB Data

All previous cells used synthetic coordinates. Now we work with real protein structures. We'll download Ubiquitin (1UBQ) from RCSB, compute RMSD against a known structure, and evaluate an ESMFold prediction.


In [ ]:
# REAL PDB DOWNLOAD AND PARSING
# We download 1UBQ (human ubiquitin, 76 residues) using BioPython
# and handle the real-world messiness: HETATM records, multiple chains, missing residues

import numpy as np
import urllib.request
import io

def fetch_pdb(pdb_id):
    """Download a PDB structure from RCSB."""
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            content = r.read().decode('utf-8')
        print(f"Downloaded {pdb_id.upper()}.pdb ({len(content)} bytes)")
        return content
    except Exception as e:
        print(f"Network unavailable ({e}) — using hardcoded 1UBQ excerpt")
        return None

def parse_ca_coords(pdb_content, chain='A'):
    """Extract C-alpha coordinates from PDB ATOM records."""
    coords = {}
    residue_names = {}
    for line in pdb_content.split('\n'):
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            if line[21] == chain or chain == '*':
                res_num = int(line[22:26].strip())
                res_name = line[17:20].strip()
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords[res_num] = np.array([x, y, z])
                residue_names[res_num] = res_name
    return coords, residue_names

# Hardcoded first 20 Cα coordinates of 1UBQ (angstroms)
# Source: PDB 1UBQ, chain A (76 residue human ubiquitin)
UBIQUITIN_EXCERPT = """ATOM      5  CA  MET A   1       1.207  -2.658  -0.000  1.00 21.27           C
ATOM     14  CA  GLN A   2       3.256  -3.929  -2.849  1.00 16.52           C
ATOM     22  CA  ILE A   3       2.753  -7.528  -3.025  1.00 13.08           C
ATOM     32  CA  PHE A   4       3.948  -7.920  -6.613  1.00 11.36           C
ATOM     42  CA  VAL A   5       3.318  -4.607  -7.818  1.00  9.78           C
ATOM     49  CA  LYS A   6       4.867  -4.153 -11.079  1.00 13.23           C
ATOM     58  CA  THR A   7       4.559  -0.440 -11.562  1.00  9.84           C
ATOM     65  CA  LEU A   8       6.462  -0.227  -8.621  1.00  8.74           C
ATOM     73  CA  THR A   9       9.843  -1.274  -9.485  1.00  9.33           C
ATOM     80  CA  GLY A  10      10.986  -1.289  -6.024  1.00  9.97           C
ATOM     84  CA  LYS A  11      10.028   2.298  -5.266  1.00 10.20           C
ATOM     93  CA  THR A  12      11.912   3.350  -2.435  1.00 10.16           C
ATOM    100  CA  ILE A  13      13.082   0.018  -1.452  1.00  9.33           C
ATOM    108  CA  THR A  14      13.987   1.009   1.966  1.00  9.93           C
ATOM    115  CA  LEU A  15      12.685   4.367   2.268  1.00 10.27           C
ATOM    123  CA  GLU A  16      11.619   5.062   5.667  1.00 11.03           C
ATOM    131  CA  VAL A  17      11.950   8.772   5.695  1.00 10.54           C
ATOM    138  CA  GLU A  18      10.534   9.979   8.716  1.00 10.66           C
ATOM    146  CA  PRO A  19       7.129   8.994   9.119  1.00 11.39           C
ATOM    151  CA  SER A  20       6.148  11.782  11.200  1.00 12.21           C"""

pdb_text = fetch_pdb('1UBQ')
if pdb_text is None:
    pdb_text = UBIQUITIN_EXCERPT

coords, res_names = parse_ca_coords(pdb_text)
print(f"\nParsed {len(coords)} Cα atoms from chain A")
print("Real-world PDB parsing issues:")
print("  - HETATM records (ligands, water) → must be excluded from protein analysis")
print("  - Multiple conformations (altLoc field) → must choose one (usually altLoc A)")
print("  - Missing residues → gaps that break sequence numbering")
print("  - Insertion codes (e.g., 'A123A') → antibody variable region insertions")

# Display residues found
residue_list = [(k, res_names[k]) for k in sorted(coords.keys())]
print(f"\nFirst 10 residues: {residue_list[:10]}")
print(f"CA coordinates of residue 1 (MET): {coords[min(coords.keys())]}")

In [ ]:
# RMSD COMPUTATION ON REAL STRUCTURE + SIMULATED PREDICTION

# Use actual 1UBQ Cα coordinates (first 20 residues)
res_ids = sorted(coords.keys())[:20]
ref_coords = np.array([coords[r] for r in res_ids])

# Simulate a "prediction" with known error (as if from AlphaFold)
np.random.seed(42)
rotation_angle = 0.05  # 3 degrees
noise_per_residue = 0.8  # Angstroms RMSD per residue

# Small rotation matrix (around z-axis)
c, s = np.cos(rotation_angle), np.sin(rotation_angle)
R_sim = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

pred_coords = ref_coords @ R_sim.T + np.random.normal(0, noise_per_residue*0.5,
                                                        ref_coords.shape)

# Kabsch alignment (optimal superposition)
def kabsch(P, Q):
    """Align P onto Q using Kabsch algorithm. Returns aligned P."""
    P_c = P - P.mean(axis=0)
    Q_c = Q - Q.mean(axis=0)
    H = P_c.T @ Q_c
    U, S, Vt = np.linalg.svd(H)
    # Correct for reflection
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1, 1, d])
    R = Vt.T @ D @ U.T
    t = Q.mean(axis=0) - R @ P.mean(axis=0)
    return P @ R.T + t, R, t

pred_aligned, R, t = kabsch(pred_coords, ref_coords)
rmsd_after = np.sqrt(np.mean(np.sum((pred_aligned - ref_coords)**2, axis=1)))
rmsd_before = np.sqrt(np.mean(np.sum((pred_coords - ref_coords)**2, axis=1)))

print(f"RMSD before Kabsch alignment: {rmsd_before:.3f} Å")
print(f"RMSD after Kabsch alignment:  {rmsd_after:.3f} Å")
print()
print("RMSD interpretation for proteins:")
print("  < 0.5 Å : Near-perfect prediction (better than experimental resolution for most)")
print("  0.5–2.0 Å : Good prediction, backbone correct")
print("  2.0–4.0 Å : Approximate, some secondary structure elements displaced")
print("  > 4.0 Å : Poor prediction, only topology may be correct")
print()
print("For reference:")
print("  AlphaFold2 achieves < 1.5 Å RMSD on most single-domain proteins")
print("  NMR vs X-ray structures of the same protein: typically 0.5–1.5 Å RMSD")
print("  B-factor (crystallographic): high B-factor residues have large RMSD between structures")

# Per-residue RMSD
per_res_rmsd = np.sqrt(np.sum((pred_aligned - ref_coords)**2, axis=1))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(per_res_rmsd)), per_res_rmsd, color='steelblue', alpha=0.8)
ax.axhline(rmsd_after, color='red', linestyle='--', label=f'Overall RMSD = {rmsd_after:.2f} Å')
ax.set_xlabel('Residue index')
ax.set_ylabel('Per-residue RMSD (Å)')
ax.set_title('Per-Residue RMSD: Simulated Prediction vs 1UBQ')
ax.legend()
ax.set_ylim(0, None)
plt.tight_layout()
plt.savefig('per_residue_rmsd.png', dpi=100)
plt.show()

## 🌐 Structure Visualization (in Jupyter)

To visualize protein structures interactively in Jupyter, use `py3Dmol` or `nglview`:

```python
# Option 1: py3Dmol (works in Jupyter, lighter weight)
# pip install py3Dmol
import py3Dmol
view = py3Dmol.view(query='pdb:1UBQ')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.show()

# Option 2: nglview (full-featured, works in JupyterLab)
# pip install nglview
import nglview as nv
view = nv.show_pdbid('1UBQ')
view.add_representation('cartoon', color='residueindex')
view

# Option 3: For a saved PDB file:
# view = nv.show_file('my_structure.pdb')
```

**Exercise:** After installing py3Dmol, load 1UBQ and color it by secondary structure. Identify the beta-grasp fold (ubiquitin's signature). Hover over residue Lys48 — this is where polyubiquitin chains form for proteasomal degradation.


In [ ]:
# EVALUATE A REAL ESMFold PREDICTION (via API)
# ESMFold is free and publicly accessible via Meta's API

import json

def query_esmfold(sequence):
    """Query ESMFold API to get a predicted PDB structure."""
    import urllib.request
    url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
    data = sequence.encode('utf-8')
    req = urllib.request.Request(url, data=data,
                                  headers={'Content-Type': 'application/x-www-form-urlencoded'})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.read().decode('utf-8')
    except Exception as e:
        return None

# Ubiquitin sequence (human, 76 aa)
UBIQUITIN_SEQ = ("MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG")

print("ESMFold API query for human ubiquitin...")
pdb_predicted = query_esmfold(UBIQUITIN_SEQ)

if pdb_predicted and 'ATOM' in pdb_predicted:
    pred_coords_esm, _ = parse_ca_coords(pdb_predicted)
    print(f"ESMFold prediction: {len(pred_coords_esm)} Cα atoms parsed")

    # Compare predicted vs 1UBQ
    common_res = sorted(set(coords.keys()) & set(pred_coords_esm.keys()))
    if len(common_res) > 10:
        ref_arr = np.array([coords[r] for r in common_res])
        pred_arr = np.array([pred_coords_esm[r] for r in common_res])
        pred_aln, _, _ = kabsch(pred_arr, ref_arr)
        rmsd = np.sqrt(np.mean(np.sum((pred_aln - ref_arr)**2, axis=1)))
        print(f"\nESMFold vs 1UBQ RMSD: {rmsd:.3f} Å (on {len(common_res)} Cα atoms)")
        print("Expected: ESMFold achieves ~1-2 Å RMSD on well-folded single-domain proteins")
    else:
        print("Not enough common residues for comparison")
else:
    print("ESMFold API unavailable (may need internet access).")
    print("Expected result: ESMFold predicts ubiquitin at ~1.2 Å RMSD from 1UBQ")
    print("\nAlternative: use the ESMFold web interface at https://esmatlas.com/")
    print("  1. Paste the sequence above")
    print("  2. Click 'Predict'")
    print("  3. Download the PDB file")
    print("  4. Parse it with parse_ca_coords() above and compute RMSD vs 1UBQ")
    print()
    print("Simulating expected output:")
    simulated_rmsd = 1.2 + np.random.normal(0, 0.1)
    print(f"  Simulated ESMFold RMSD: {simulated_rmsd:.2f} Å")
    print(f"  Interpretation: {'Excellent' if simulated_rmsd < 1.5 else 'Good'} prediction")

## 🐛 Debug Exercise — RMSD, Kabsch Alignment, TM-score

Find and fix the **3 bugs** in the structural alignment code below.

**Expected correct output:**
```
Centroid-corrected RMSD: ~0.00  (identical structures after superposition)
Kabsch rotation det: +1.0  (proper rotation, not reflection)
TM-score: value between 0 and 1
```

<details>
<summary>Show answer</summary>

**Bug 1 — RMSD without centering:** RMSD should compare structures after subtracting centroids.
Fix: `rmsd = np.sqrt(np.mean(np.sum((P - P.mean(0) - (Q - Q.mean(0)))**2, axis=1)))`.

**Bug 2 — Kabsch missing SVD sign correction:** SVD can return a reflection (det(VU^T) = -1).
Fix: after SVD, check `if np.linalg.det(V @ U.T) < 0: S_diag[-1] *= -1`.

**Bug 3 — TM-score wrong denominator:** TM-score normalizes by L0 (the length of the *target*
or a reference length), not N (the number of aligned pairs). When all residues are aligned
they're equal, but in partial alignments using N gives a biased score.
Fix: use `L0 = len(Q)` as the denominator.
</details>


In [ ]:
# DEBUG EXERCISE — RMSD, Kabsch, TM-score — find and fix 3 bugs
import numpy as np

np.random.seed(0)
N = 50
# Two identical structures with a known offset (translation)
P = np.random.randn(N, 3)
Q = P + np.array([5.0, 3.0, -2.0])   # translated copy

# BUG 1: RMSD computed without centering structures first
# For superposition quality, subtract centroids before computing RMSD
rmsd_naive = np.sqrt(np.mean(np.sum((P - Q)**2, axis=1)))
print(f"Naive RMSD (includes translation): {rmsd_naive:.3f}")
print("Expected after centering: ~0.00")

# Kabsch algorithm
def kabsch(P, Q):
    Pc = P - P.mean(0)
    Qc = Q - Q.mean(0)
    H = Pc.T @ Qc
    U, S_vals, Vt = np.linalg.svd(H)
    V = Vt.T

    # BUG 2: missing determinant sign correction
    # If det(V @ U.T) < 0, we have a reflection, not a proper rotation
    # Fix: S_diag = [1, 1, 1]; if det < 0: S_diag[-1] = -1; R = V @ diag(S_diag) @ U.T
    R = V @ U.T   # may produce improper rotation (reflection) for some inputs
    return R, Pc, Qc

R, Pc, Qc = kabsch(P, Q)
det = np.linalg.det(R)
print(f"Kabsch rotation matrix det: {det:.4f}  (should be +1.0 for proper rotation)")
P_rot = Pc @ R.T
rmsd_kabsch = np.sqrt(np.mean(np.sum((P_rot - Qc)**2, axis=1)))
print(f"RMSD after Kabsch: {rmsd_kabsch:.4f}")

# TM-score
def tm_score(P, Q, d0=2.0):
    Pc = P - P.mean(0)
    Qc = Q - Q.mean(0)
    dists = np.sqrt(np.sum((Pc - Qc)**2, axis=1))
    N_aligned = len(P)

    # BUG 3: should normalize by L0 = len(Q) (target length), not N_aligned
    # When N_aligned == len(Q) they're the same, but conceptually L0 is the reference
    score = (1.0 / N_aligned) * np.sum(1.0 / (1.0 + (dists / d0)**2))
    return score

tm = tm_score(P, Q)
print(f"TM-score: {tm:.4f}")


## Notebook Complete

**You can now:**
- Parse PDB files with BioPython and extract residue-level 3D coordinates
- Compute RMSD between structures and apply the Kabsch alignment algorithm

**Where this fits:**
- Previous: 02_genomics_gene_analysis
- Next: 02_structure_quality_and_features — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes